# AdaptTutor User Study Evaluation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ast
import json

# Load datasets
sessions = pd.read_csv('../backend/data_exports/sessions.csv')
telemetry = pd.read_csv('../backend/data_exports/telemetry.csv')
surveys = pd.read_csv('../backend/data_exports/surveys.csv')

# Clean up dates
sessions['start_time'] = pd.to_datetime(sessions['start_time'])
sessions['end_time'] = pd.to_datetime(sessions['end_time'])

telemetry['timestamp'] = pd.to_datetime(telemetry['timestamp'])
telemetry['event_data'] = telemetry['event_data'].apply(lambda x: json.loads(x) if pd.notnull(x) else {})

print(f"Loaded {len(sessions)} sessions, {len(telemetry)} telemetry events, and {len(surveys)} survey responses.")

## 1. Completion Times
Calculate the average time spent from session start to completion for the Adaptive vs Static groups.

In [ ]:
# Only consider sessions that actually finished (end_time is not null)
completed_sessions = sessions.dropna(subset=['end_time']).copy()
completed_sessions['duration_minutes'] = (completed_sessions['end_time'] - completed_sessions['start_time']).dt.total_seconds() / 60

avg_times = completed_sessions.groupby('condition')['duration_minutes'].mean().reset_index()
print(avg_times)

plt.figure(figsize=(8, 5))
plt.bar(avg_times['condition'], avg_times['duration_minutes'], color=['#3b82f6', '#ef4444'])
plt.title('Average Completion Time by Condition')
plt.ylabel('Minutes')
plt.show()

## 2. Telemetry: Frustration Metrics
Look at the frequency of syntax errors and 'run' attempts.

In [ ]:
# Merge telemetry with conditions
events_df = telemetry.merge(sessions[['session_id', 'condition']], on='session_id')

# Count runs per session
run_events = events_df[events_df['event_type'] == 'run_click']
runs_per_session = run_events.groupby(['condition', 'session_id']).size().reset_index(name='run_count')

avg_runs = runs_per_session.groupby('condition')['run_count'].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.bar(avg_runs['condition'], avg_runs['run_count'], color=['#3b82f6', '#ef4444'])
plt.title('Average Number of Code Executions (Runs) per Session')
plt.ylabel('Count')
plt.show()

## 3. Survey Results
Analyze the subjective 'Usefulness' Likert scale from the post-survey.

In [ ]:
post_surveys = surveys[surveys['survey_type'] == 'post'].copy()
post_surveys['responses'] = post_surveys['responses'].apply(lambda x: json.loads(x) if pd.notnull(x) else {})

# Extract 'usefulness' score if they answered it
post_surveys['usefulness_score'] = post_surveys['responses'].apply(lambda x: int(x.get('usefulness', 0)))
post_surveys = post_surveys[post_surveys['usefulness_score'] > 0] # Filter empty

survey_with_cond = post_surveys.merge(sessions[['session_id', 'condition']], on='session_id')

avg_usefulness = survey_with_cond.groupby('condition')['usefulness_score'].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.bar(avg_usefulness['condition'], avg_usefulness['usefulness_score'], color=['#3b82f6', '#ef4444'])
plt.title('Average Perceived Usefulness (1-5 Scale)')
plt.ylabel('Score')
plt.ylim(0, 5)
plt.show()